**1. Bookings and revenue by city**

In [0]:
SELECT city,
       COUNT(*) AS total_bookings,
       ROUND(AVG(total_paid_usd), 2) AS avg_booking_value,
       ROUND(SUM(total_paid_usd), 2) AS total_revenue
FROM airbnb.bookings
WHERE booking_status = 'Completed'
GROUP BY city
ORDER BY total_revenue DESC;

city,total_bookings,avg_booking_value,total_revenue
Miami,1217,1040.24,1265966.48
New York City,1607,753.56,1210967.48
Los Angeles,1289,763.28,983867.72
Chicago,915,706.35,646313.16
Nashville,805,694.35,558950.08
Austin,631,880.88,555835.26
Portland,750,498.4,373800.5


**2. Cancellation rate by city**

In [0]:
SELECT city,
       COUNT(*) AS total_bookings,
       SUM(CASE WHEN booking_status = 'Cancelled by guest' THEN 1 ELSE 0 END) AS cancellations,
       ROUND(100.0 * SUM(CASE WHEN booking_status = 'Cancelled by guest' THEN 1 ELSE 0 END) / COUNT(*), 1) AS cancellation_rate_pct
FROM airbnb.bookings
GROUP BY city
ORDER BY cancellation_rate_pct DESC;

city,total_bookings,cancellations,cancellation_rate_pct
Austin,890,140,15.7
Portland,1040,159,15.3
Los Angeles,1791,263,14.7
Chicago,1278,180,14.1
New York City,2229,310,13.9
Nashville,1122,154,13.7
Miami,1650,201,12.2


**Which markets generate the most revenue?**

Query 1 — Revenue by city


In [0]:
SELECT city,
       COUNT(*) AS total_bookings,  
       ROUND(AVG(total_paid_usd), 2) AS avg_booking_value,
       ROUND(SUM(total_paid_usd), 0) AS total_revenue
FROM airbnb.bookings
WHERE booking_status = 'Completed'
GROUP BY city
ORDER BY total_revenue DESC;

city,total_bookings,avg_booking_value,total_revenue
Miami,1217,1040.24,1265966.0
New York City,1607,753.56,1210967.0
Los Angeles,1289,763.28,983868.0
Chicago,915,706.35,646313.0
Nashville,805,694.35,558950.0
Austin,631,880.88,555835.0
Portland,750,498.4,373801.0


**Which cities have retention problems?**

Query 2 — Cancellation rate by city

In [0]:
SELECT city,
       COUNT(*) AS total_bookings,
       SUM(CASE WHEN booking_status LIKE 'Cancelled%' THEN 1 ELSE 0 END) AS cancellations,
       ROUND(100.0 * SUM(CASE WHEN booking_status LIKE 'Cancelled%' THEN 1 ELSE 0 END) / COUNT(*), 1) AS cancellation_rate_pct
FROM airbnb.bookings
GROUP BY city
ORDER BY cancellation_rate_pct DESC;
       

city,total_bookings,cancellations,cancellation_rate_pct
Austin,890,168,18.9
Chicago,1278,237,18.5
Portland,1040,190,18.3
Los Angeles,1791,326,18.2
Nashville,1122,202,18.0
New York City,2229,401,18.0
Miami,1650,267,16.2


**Do superhosts actually earn more?**

Query 3 — Superhost revenue premium


In [0]:
SELECT h.superhost,
        COUNT(b.booking_id) AS total_bookings,
        ROUND(AVG(b.total_paid_usd), 2) AS avg_booking_value,
        ROUND(AVG(l.nightly_price_usd), 2) AS avg_nightly_rate
FROM airbnb.bookings b 
JOIN airbnb.listings l ON b.listing_id = l.listing_id
JOIN airbnb.hosts h ON l.host_id = h.host_id
WHERE b.booking_status = 'Completed'
GROUP BY h.superhost;

superhost,total_bookings,avg_booking_value,avg_nightly_rate
0,5360,755.36,129.83
1,1854,834.4,137.53


**Do last-minute bookers cancel more often?**

Query 4 — Booking lead time vs cancellation

In [0]:
SELECT
  CASE 
      WHEN booked_days_in_advance <= 7 THEN '0-7 days'
      WHEN booked_days_in_advance <= 30 THEN '8-30 days'
      WHEN booked_days_in_advance <= 90 THEN '31-90 days'
      ELSE '90+ days'
  END AS lead_time_bucket,
  COUNT(*) AS total_bookings,
  ROUND(100.0 * SUM(CASE WHEN booking_status LIKE 'Cancelled%' THEN 1 ELSE 0 END) / COUNT(*), 1) AS cancellation_rate_pct
FROM airbnb.bookings
GROUP BY lead_time_bucket
ORDER BY MIN(booked_days_in_advance);


lead_time_bucket,total_bookings,cancellation_rate_pct
0-7 days,361,18.6
8-30 days,1280,17.0
31-90 days,3278,17.4
90+ days,5081,18.4


**Where does visitor money actually go in each city?**

Query 5 — Local economic impact by city

In [0]:
SELECT city,
       spend_category,
       ROUND(SUM(amount_usd), 0) AS total_spend,
       ROUND(AVG(amount_usd), 2) AS avg_spend_per_stay,
       COUNT(DISTINCT booking_id) AS unique_stays
FROM airbnb.spending
GROUP BY city, spend_category
ORDER BY city, total_spend DESC;

city,spend_category,total_spend,avg_spend_per_stay,unique_stays
Austin,Dining & Restaurants,1062469.0,1963.9,541
Austin,Local Transportation,679894.0,1327.92,512
Austin,Shopping & Retail,653572.0,1933.64,338
Austin,Attractions & Activities,647612.0,1545.61,419
Austin,Tours & Experiences,466790.0,2121.77,220
Austin,Groceries & Markets,441864.0,977.57,452
Austin,Nightlife & Entertainment,251180.0,947.85,265
Austin,Spa & Wellness,197954.0,1663.48,119
Chicago,Dining & Restaurants,1795803.0,2244.75,800
Chicago,Shopping & Retail,1345516.0,2696.42,499


**Which neighborhoods are hotspots within each city?**

Query 6 — Top neighborhoods by occupancy


In [0]:
SELECT l.city,
       l.neighborhood,
       COUNT(b.booking_id) AS total_bookings,
       ROUND(AVG(b.total_paid_usd), 2) AS avg_booking_value,
       ROUND(AVG(l.listing_rating), 2) AS avg_listing_rating
FROM airbnb.listings l
JOIN airbnb.bookings b ON l.listing_id = b.listing_id
WHERE b.booking_status = 'Completed'
GROUP BY l.city, l.neighborhood
ORDER BY l.city, total_bookings DESC;

city,neighborhood,total_bookings,avg_booking_value,avg_listing_rating
Austin,Hyde Park,154,953.89,4.09
Austin,South Congress,131,1026.66,4.08
Austin,Rainey Street,128,569.03,4.01
Austin,Downtown,128,608.49,4.05
Austin,East Austin,90,1374.68,3.99
Chicago,River North,218,834.61,4.1
Chicago,Logan Square,197,746.51,4.14
Chicago,Wicker Park,174,636.76,3.97
Chicago,Lincoln Park,164,472.23,4.11
Chicago,South Loop,162,796.69,3.95


**Revenue per guest (spend efficiency)**

In [0]:
SELECT city,
       ROUND(SUM(amount_usd) / COUNT(DISTINCT booking_id), 2) AS spend_per_stay,
       ROUND(SUM(amount_usd) / SUM(s.nights_stayed), 2) AS spend_per_night
FROM airbnb.spending s
GROUP BY city
ORDER BY spend_per_night DESC;

city,spend_per_stay,spend_per_night
New York City,6592.4,365.08
Miami,8447.93,361.46
Chicago,8286.98,349.8
Los Angeles,6978.19,339.56
Nashville,6621.76,313.91
Portland,5910.58,291.62
Austin,6975.17,291.61


**Travel purpose vs avg spend**

In [0]:
SELECT travel_purpose,
       ROUND(AVG(amount_usd), 2) AS avg_spend,
       COUNT(DISTINCT booking_id) AS stays
FROM airbnb.spending
GROUP BY travel_purpose
ORDER BY avg_spend DESC;

travel_purpose,avg_spend,stays
Events/Festival,2015.15,559
Business,1852.05,1429
Family vacation,1730.54,1156
Leisure,1473.91,3028
Remote work,1216.47,768
Medical,1160.03,120
Relocation,822.73,152
